In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import sys

# Load the data we just scraped and cleaned
sys.path.append('../src')
from data_cleaning import clean_listings
from feature_engineering import engineer_features

# Load and clean (assuming you have a raw_listings.csv in your data folder)
df = pd.read_csv('../data/raw_listings.csv')
df = clean_listings(df)

# Engineer features (e.g., using coordinates for downtown Seattle)
df = engineer_features(df, city_center_lat=47.6062, city_center_lon=-122.3321)

df.head()

In [ ]:
# Set a clean visual style
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.histplot(df['price'], bins=50, kde=True, color='teal')
plt.title('Distribution of Rental Prices', fontsize=16, fontweight='bold')
plt.xlabel('Monthly Rent ($)')
plt.ylabel('Count of Listings')
plt.show()

# Insight: If this is heavily right-skewed, our XGBoost model will handle it well, 
# but a linear regression model would have needed a log transformation here!

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df, 
    x='dist_to_center_km', 
    y='price', 
    scatter_kws={'alpha':0.3, 'color': 'coral'}, 
    line_kws={'color':'red', 'linewidth': 2}
)
plt.title('Does distance from downtown lower the rent?', fontsize=16)
plt.xlabel('Distance to City Center (km)')
plt.ylabel('Monthly Rent ($)')
plt.show()

In [ ]:
# Plotly renders an interactive map where you can hover over dots to see details.
# This is O(N) rendering and handles thousands of points easily.
fig = px.scatter_mapbox(
    df, 
    lat="latitude", 
    lon="longitude", 
    color="price",
    size="bedrooms",
    color_continuous_scale=px.colors.sequential.Plasma,
    size_max=15, 
    zoom=10,
    hover_data=["price", "bedrooms", "bathrooms", "neighborhood"],
    title="RentRadar Interactive Heatmap"
)

# You will need a Mapbox token for the background map, or use the open-source 'open-street-map'
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()